In [549]:
# Code to map Maven Hepatitis elements to Origami Hepatitis elements
# Using semantic similarity library
# Author: Sandeep Chintabathina
# April 2026

In [550]:
# For each Maven Hepatitis data element, check if there is an exact or close matching data element in the origami Hepatitis package
# The output will list the maven element alongside the origami element and other supporting attributes

In [551]:
from sentence_transformers import SentenceTransformer


In [552]:
import pandas as pd
import numpy as np

In [553]:
# Read maven elements
df_maven = pd.read_csv('output/maven_all_with_mmg_and_cste.csv',na_filter=False)
len(df_maven)

9501

In [554]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9501 entries, 0 to 9500
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Model                  9501 non-null   object
 1   question_number        9501 non-null   int64 
 2   depth                  9501 non-null   int64 
 3   Question               9501 non-null   object
 4   Question.Package.ID    9501 non-null   object
 5   Question.Package.Name  9501 non-null   object
 6   Message                9501 non-null   object
 7   mmg_element            9501 non-null   object
 8   cste_element           9501 non-null   object
dtypes: int64(2), object(7)
memory usage: 668.2+ KB


In [555]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If space, digits, uppercase letters, lowercase letters, or underscore, add it to new name otherwise ignore
        if (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=ch
    return new_name.strip()

In [556]:
clean_up('[sdfs] 34_AS ^#~')

'sdfs 34_AS'

In [557]:
# Clean up questions or data elements
df_maven.loc[:,'Question'] =[ clean_up(q) for q in df_maven.loc[:,'Question']]

In [558]:
df_maven.loc[:,'Model'].value_counts(dropna=False)

Model
General                  3866
STD                      2425
Coinfection               923
Hepatitis                 901
Childhood Lead            606
Cluster                   425
Lead investigation        255
STD EHR and screening      85
Aggregate                  15
Name: count, dtype: int64

In [559]:
df_maven_hep = df_maven[df_maven['Model']=='Hepatitis']
len(df_maven_hep)

901

In [560]:
# No duplicates
df_maven_hep[df_maven_hep.duplicated(['Question'],keep=False)]

,Model,question_number,depth,Question,Question.Package.ID,Question.Package.Name,Message,mmg_element,cste_element


In [561]:
# Read origami elements
df_hep = pd.read_csv('origami/csv/Custom__CaseHEP.csv',na_filter=False)
len(df_hep)

512

In [562]:
df_hep.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 512 entries, 0 to 511
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Field                 512 non-null    object
 1   Type                  512 non-null    object
 2   Label                 512 non-null    object
 3   Required in Database  512 non-null    bool  
 4   Can't Protect         512 non-null    bool  
 5   FK Table              512 non-null    object
 6   CodeTypeID            512 non-null    object
 7   Static Codes          512 non-null    object
 8   Field Notes           512 non-null    object
 9   Global Comments       512 non-null    object
 10  Local Comments        512 non-null    object
dtypes: bool(2), object(9)
memory usage: 37.1+ KB


In [563]:
# treatment elements
df_treatment = pd.read_csv('origami/csv/Custom__Treatment.csv',na_filter=False)
len(df_treatment)

311

In [564]:
df_altliver = pd.read_csv('origami/csv/Custom__ALTLiverEnzyme.csv',na_filter=False)
len(df_altliver)

48

In [565]:
df_astliver = pd.read_csv('origami/csv/Custom__ASTLiveEnzyme.csv',na_filter=False)
len(df_astliver)

48

In [566]:
df_hep.loc[:,'Label'].value_counts(dropna=False)

Label
                                                     177
[INV654]Year the patient received the most recent      3
[INV887]Does subject have diabetes                     2
[LP38350-2]Hepatitis E virus Ab                        2
[LP38352-8]Hepatitis E virus Ab IgM                    2
                                                    ... 
Clinical review/entry completed                        1
Clinical State                                         1
Clinic City                                            1
Clinic mailing address (line 2)                        1
Hep A Subtype Result                                   1
Name: count, Length: 321, dtype: int64

In [567]:
# Get the two columns of interest and non-blank values
df_hep = df_hep[['Field','Label']]
df_hep = df_hep[df_hep['Label']!='']
len(df_hep)

335

In [568]:
df_hep = df_hep.reset_index(drop=True)

In [569]:
df_hep

,Field,Label
0,CaseHEPID,Case HEPID
1,ClientID,Client
2,EntryUserID,Entry User
3,EntryDate,Entry Date
4,ModifiedUserID,Modified User
...,...,...
330,CustomCode164ID,Was the patient a household contact of a child or
331,CustomCode165ID,Transmission to Infant from Mother
332,CustomCode166ID,INV655_medicalcare
333,CustomCode167ID,Hep A RNA (PCR/NAAT)


In [570]:
# Original label to be used later for output
df_hep.rename(columns={'Label':'original_label'},inplace=True)
df_hep.columns

Index(['Field', 'original_label'], dtype='object')

In [571]:
# Process the column values (eliminate the values in square brackets)
df_hep.loc[:,'Label'] = [ label.split(']')[1].strip() if ']' in label else label.strip() for label in df_hep.loc[:,'original_label']]

In [572]:
# Clean up column names in both data sets
df_hep.loc[:,'Label'] =[ clean_up(q) for q in df_hep.loc[:,'Label']]

In [573]:
df_hep.loc[:,'Label'].value_counts()

Label
Specimen Collection Date                     21
Year the patient received the most recent     3
TREATING_PHYSICIAN_MRN                        2
Does subject have diabetes                    2
Reason for Testing                            2
                                             ..
Childcare Location Specify                    1
Hepatitis B virus core Ab                     1
Hepatitis B virus Core IgM                    1
Hepatitis B virus DNA                         1
Hep A Subtype Result                          1
Name: count, Length: 298, dtype: int64

In [574]:
# duplicates labels exist 
df_hep[df_hep.duplicated(['Label'],keep=False)]

,Field,original_label,Label
9,CustomCode3ID,[INV578]Was the subject jaundiced,Was the subject jaundiced
10,CustomCode4ID,[INV887]Does subject have diabetes,Does subject have diabetes
14,CustomCode8ID,[LP38351-0]Hepatitis E virus Ab IgG,Hepatitis E virus Ab IgG
15,CustomCode9ID,[LP38352-8]Hepatitis E virus Ab IgM,Hepatitis E virus Ab IgM
17,CustomDate2,[LP16708-7_dt]Specimen Collection Date,Specimen Collection Date
18,CustomDate3,[LP38316-3_dt]Specimen Collection Date,Specimen Collection Date
19,CustomDate4,[LP38318-9_dt]Specimen Collection Date,Specimen Collection Date
20,CustomDate5,[LP38351-0_dt]Specimen Collection Date,Specimen Collection Date
21,CustomDate6,[LP38352-8_dt]Specimen Collection Date,Specimen Collection Date
22,CustomDate7,[LP418768-0_dt]Specimen Collection Date,Specimen Collection Date


In [575]:
# dropping them
df_hep = df_hep.drop_duplicates(['Label'])

In [576]:
df_hep[df_hep.duplicated(['Label'],keep=False)]

,Field,original_label,Label


In [579]:
df_hep = df_hep.reset_index(drop=True)
df_hep

,Field,original_label,Label
0,CaseHEPID,Case HEPID,Case HEPID
1,ClientID,Client,Client
2,EntryUserID,Entry User,Entry User
3,EntryDate,Entry Date,Entry Date
4,ModifiedUserID,Modified User,Modified User
...,...,...,...
293,CustomCode164ID,Was the patient a household contact of a child or,Was the patient a household contact of a child or
294,CustomCode165ID,Transmission to Infant from Mother,Transmission to Infant from Mother
295,CustomCode166ID,INV655_medicalcare,INV655_medicalcare
296,CustomCode167ID,Hep A RNA (PCR/NAAT),Hep A RNA PCRNAAT


In [491]:
# Do a semantic match between Maven Hep elements and Origami Hep labels

In [580]:
maven_elements = [str.replace(maven_de,'_',' ') for maven_de in df_maven_hep.Question]
origami_elements = [str.replace(col,'_',' ') for col in df_hep.Label]

In [493]:
#origami_elements

In [581]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [582]:
# Compute embeddings
embeddings_maven = model.encode(maven_elements)
embeddings_origami = model.encode(origami_elements)

In [583]:
print(len(embeddings_origami))
print(len(embeddings_maven))
embeddings_origami
embeddings_maven

298
901


array([[-0.04004316,  0.03481774, -0.02303074, ...,  0.06413301,
         0.07289726,  0.05008497],
       [ 0.01921209,  0.03713002,  0.04779667, ...,  0.04271896,
        -0.05558128, -0.02244971],
       [-0.1053156 ,  0.01508899,  0.01699461, ...,  0.08866485,
        -0.01472143, -0.02046957],
       ...,
       [-0.08527962,  0.04394535, -0.00333766, ..., -0.00748747,
         0.13976483, -0.05983388],
       [-0.10253949,  0.03828218, -0.01935591, ..., -0.01946716,
         0.04943707,  0.04164878],
       [-0.04428944,  0.05000523, -0.03245968, ..., -0.02810097,
        -0.05133276,  0.00550872]], dtype=float32)

In [584]:
# Compute cosine similarities
similarities = model.similarity(embeddings_maven,embeddings_origami)

In [585]:
similarities

tensor([[ 0.1961,  0.2485,  0.1516,  ...,  0.2660,  0.1456,  0.1117],
        [ 0.1171,  0.1044,  0.1415,  ...,  0.1370,  0.0659,  0.0116],
        [ 0.0729,  0.0654,  0.0365,  ...,  0.1433,  0.0364,  0.0170],
        ...,
        [ 0.1655,  0.1939,  0.0975,  ...,  0.3144,  0.0792,  0.0587],
        [-0.0066,  0.0415,  0.1275,  ...,  0.1359,  0.0184,  0.0296],
        [ 0.2537,  0.0944,  0.0472,  ...,  0.2289,  0.3586,  0.3829]])

In [586]:
print(type(similarities))
len(similarities)

<class 'torch.Tensor'>


901

In [587]:
df_similarities = pd.DataFrame(similarities,index=df_maven_hep.Question,columns=df_hep.original_label)

In [588]:
df_similarities.to_csv('output/similarity_scores_hep.csv')

In [502]:
#df_similarities.loc['DEVELOPMENTAL_SCREENING',:]

In [589]:
df_maven_hep = df_maven_hep.reset_index(drop=True)
df_maven_hep.columns

Index(['Model', 'question_number', 'depth', 'Question', 'Question.Package.ID',
       'Question.Package.Name', 'Message', 'mmg_element', 'cste_element'],
      dtype='object')

In [590]:
df_output = df_maven_hep[['Model','depth','Question','Message','mmg_element','cste_element']]

In [591]:
df_similarities.info()

<class 'pandas.core.frame.DataFrame'>
Index: 901 entries, DISEASE to HEPB_LEGACY_EVENT_NOTES
Columns: 298 entries, Case HEPID to Hep A Subtype Result
dtypes: float32(298)
memory usage: 1.0+ MB


In [592]:
#For every maven element (index), identify origami elements (column) that is the best match

filtered=[]
for idx in df_similarities.index:
    # Looking for columns with score higher than 0.92
    v = df_similarities.loc[idx,:]>0.75

    # Get the column with the maximum score
    max=0.0
    max_col=''
    for i,col in enumerate(df_similarities.columns):
        if v.iloc[i]==True:
            #print('value',df_similarities.loc[idx,col])
            if (df_similarities.loc[idx,col] > max):
                max = df_similarities.loc[idx,col]
                max_col=col

    #filtered.append({col:df_similarities.loc[idx,col]})
    #print(idx+','+max_col+':'+str(max)+'\n')
    filtered.append({'Question':idx,'target_label':max_col})



In [593]:
df_filtered = pd.DataFrame(filtered)
print(len(df_filtered))
#df_filtered.loc[:,'target_label'].value_counts(dropna=False)

901


In [594]:
df_merge = pd.merge(df_filtered,df_hep,left_on='target_label',right_on='original_label',how='left')

In [595]:
df_merge

,Question,target_label,Field,original_label,Label
0,DISEASE,,NaN,NaN,NaN
1,EVENT_DATE,,NaN,NaN,NaN
2,EPISODE_DATE,,NaN,NaN,NaN
3,EPISODE_DATE_BASED_ON,,NaN,NaN,NaN
4,INVESTIGATION_STATUS,Final investigation status,CustomCode152ID,Final investigation status,Final investigation status
...,...,...,...,...,...
896,LEGACY_CONTACT,,NaN,NaN,NaN
897,LEGACY_PHC,,NaN,NaN,NaN
898,LEGACY_LAB,,NaN,NaN,NaN
899,LEGACY_FORM_DATA,,NaN,NaN,NaN


In [608]:
# merge the above merged df with output dataframe
df_final = pd.merge(df_output,df_merge,on='Question')
#df_final

In [609]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 901 entries, 0 to 900
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Model           901 non-null    object
 1   depth           901 non-null    int64 
 2   Question        901 non-null    object
 3   Message         901 non-null    object
 4   mmg_element     901 non-null    object
 5   cste_element    901 non-null    object
 6   target_label    901 non-null    object
 7   Field           154 non-null    object
 8   original_label  154 non-null    object
 9   Label           154 non-null    object
dtypes: int64(1), object(9)
memory usage: 70.5+ KB


In [610]:
# Add, rename, remove columns as needed
df_final['target_table'] = ['' if f=='' else 'Custom__CaseHEP' for f in df_final.target_label]
df_final=df_final.rename(columns={'Field':'target_field'})
df_final=df_final.drop('Label',axis=1)


In [611]:
df_final = df_final[["Model",'Question','target_table','target_field','target_label','mmg_element','cste_element','Message','depth']]
#df_final

In [612]:
df_final

,Model,Question,target_table,target_field,target_label,mmg_element,cste_element,Message,depth
0,Hepatitis,DISEASE,,NaN,,,,,0
1,Hepatitis,EVENT_DATE,,NaN,,,,,0
2,Hepatitis,EPISODE_DATE,,NaN,,,,Episode date,0
3,Hepatitis,EPISODE_DATE_BASED_ON,,NaN,,,,Episode date based on,0
4,Hepatitis,INVESTIGATION_STATUS,Custom__CaseHEP,CustomCode152ID,Final investigation status,,,,0
...,...,...,...,...,...,...,...,...,...
896,Hepatitis,LEGACY_CONTACT,,NaN,,,,,0
897,Hepatitis,LEGACY_PHC,,NaN,,,,,0
898,Hepatitis,LEGACY_LAB,,NaN,,,,,0
899,Hepatitis,LEGACY_FORM_DATA,,NaN,,,,,0


In [516]:
#df_final[df_final['target_field'].isna()]

In [613]:
df_final.to_csv('output/maven_HEP_origami_similarity_match.csv',index=False)